# Imports & constants

In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display, Markdown

In [16]:
# paths
base_path = Path("../input")
train_csv_path = base_path / "train.csv"
test_csv_path = base_path / "test.csv"

# load dataset
train_df = pd.read_csv(train_csv_path)
test_df = pd.read_csv(test_csv_path)

# others
RERUN = False# if true, reruns all visualisations and time-consuming operations
taget_col = "diagnosed_diabetes"

# Basic Stats

In [4]:
display(train_df.head())
display(test_df.head())

,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,...,gender,ethnicity,education_level,income_level,smoking_status,employment_status,family_history_diabetes,hypertension_history,cardiovascular_history,diagnosed_diabetes
0,0,31,1,45,7.7,6.8,6.1,33.4,0.93,112,...,Female,Hispanic,Highschool,Lower-Middle,Current,Employed,0,0,0,1.0
1,1,50,2,73,5.7,6.5,5.8,23.8,0.83,120,...,Female,White,Highschool,Upper-Middle,Never,Employed,0,0,0,1.0
2,2,32,3,158,8.5,7.4,9.1,24.1,0.83,95,...,Male,Hispanic,Highschool,Lower-Middle,Never,Retired,0,0,0,0.0
3,3,54,3,77,4.6,7.0,9.2,26.6,0.83,121,...,Female,White,Highschool,Lower-Middle,Current,Employed,0,1,0,1.0
4,4,54,1,55,5.7,6.2,5.1,28.8,0.90,108,...,Male,White,Highschool,Upper-Middle,Never,Retired,0,1,0,1.0


,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,...,triglycerides,gender,ethnicity,education_level,income_level,smoking_status,employment_status,family_history_diabetes,hypertension_history,cardiovascular_history
0,700000,45,4,100,4.3,6.8,6.2,25.5,0.84,123,...,111,Female,White,Highschool,Middle,Former,Employed,0,0,0
1,700001,35,1,87,3.5,4.6,9.0,28.6,0.88,120,...,145,Female,White,Highschool,Middle,Never,Unemployed,0,0,0
2,700002,45,1,61,7.6,6.8,7.0,28.5,0.94,112,...,184,Male,White,Highschool,Low,Never,Employed,0,0,0
3,700003,55,2,81,7.3,7.3,5.0,26.9,0.91,114,...,128,Male,White,Graduate,Middle,Former,Employed,0,0,0
4,700004,77,2,29,7.3,7.6,8.5,22.0,0.83,131,...,133,Male,White,Graduate,Low,Current,Unemployed,0,0,0


In [5]:
display(Markdown(f"#### 🚀 Number of instances in train: {len(train_df)}"))
display(Markdown(f"#### 🚀 Number of instances in test: {len(test_df)}"))
display(Markdown("#### There are no null/nan values in the tables"))

#### 🚀 Number of instances in train: 700000

#### 🚀 Number of instances in test: 300000

#### There are no null/nan values in the tables

# Distributions

In [6]:
def plot_column_dist_two(
    df1: pd.DataFrame,
    df2: pd.DataFrame,
    col: str,
    base_save_path: Path,
    bins: int = 100,
    plot_in_notebook: bool = False,
    prefix_str: str = "",
    labels=("train", "test")
):
    # Create directory
    (base_save_path / "dist").mkdir(parents=True, exist_ok=True)

    # Extract cleaned data
    data1 = df1[col].dropna()
    data2 = df2[col].dropna()

    plt.figure(figsize=(10, 6))
    sns.set_theme(style="whitegrid")

    # --------------------------------------------------------
    # Numeric → overlay normalized histograms or KDEs
    # --------------------------------------------------------
    if pd.api.types.is_numeric_dtype(data1):
        sns.histplot(
            data1, bins=bins, kde=True, stat="density",
            color="dodgerblue", label=labels[0], alpha=0.5
        )
        sns.histplot(
            data2, bins=bins, kde=True, stat="density",
            color="crimson", label=labels[1], alpha=0.5
        )

        plt.xlabel(col)
        plt.ylabel("Density")
        plt.title(f"Normalized Histogram Comparison: '{col}'", fontsize=15)

    # --------------------------------------------------------
    # Categorical → normalized countplots
    # --------------------------------------------------------
    else:
        # Convert to value counts normalized
        vc1 = data1.value_counts(normalize=True)
        vc2 = data2.value_counts(normalize=True)

        # Align categories so both bars appear for same x-values
        all_categories = sorted(set(vc1.index).union(vc2.index))
        vc1 = vc1.reindex(all_categories, fill_value=0)
        vc2 = vc2.reindex(all_categories, fill_value=0)

        # Bar plot overlay (side-by-side is also possible)
        x = range(len(all_categories))
        width = 0.4

        plt.bar(
            [p - width/2 for p in x], vc1.values,
            width=width, color="dodgerblue", alpha=0.7, label=labels[0]
        )
        plt.bar(
            [p + width/2 for p in x], vc2.values,
            width=width, color="crimson", alpha=0.7, label=labels[1]
        )

        plt.xticks(x, all_categories, rotation=45, ha="right")
        plt.xlabel(col)
        plt.ylabel("Normalized Frequency")
        plt.title(f"Normalized Category Comparison: '{col}'", fontsize=15)

    # --------------------------------------------------------
    # Final layout & save/show logic
    # --------------------------------------------------------
    plt.legend()
    plt.tight_layout()

    save_path = base_save_path / "dist" / f"{prefix_str}{col}_dist_compare.png"
    plt.savefig(save_path, dpi=300)

    if plot_in_notebook:
        plt.show()
    else:
        plt.close()

In [7]:
if RERUN:
    common_cols = [x for x in train_df.columns if x in test_df.columns]
    for col in common_cols:
        plot_column_dist_two(train_df, test_df, col, Path("ign_visualisations"), plot_in_notebook=True)

### Most distributions match for both train and test features.